In [7]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [8]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, models
from PIL import Image
import matplotlib.pyplot as plt

# Load our split dataset index
df = pd.read_csv("../Dataset/dataset_index_with_splits.csv")
print(df['split'].value_counts())
print(df.head())

split
train    784
test     172
val      168
Name: count, dtype: int64
                                          image_path class_name  class_id  \
0  ../Dataset/Wheat varieties dataset/Dilkash/D12...    Dilkash         1   
1  ../Dataset/Wheat varieties dataset/Dilkash/D12...    Dilkash         1   
2  ../Dataset/Wheat varieties dataset/Dilkash/D12...    Dilkash         1   
3  ../Dataset/Wheat varieties dataset/Dilkash/D15...    Dilkash         1   
4  ../Dataset/Wheat varieties dataset/Dilkash/D15...    Dilkash         1   

  sample_id  split  
0       D12    val  
1       D12    val  
2       D12    val  
3       D15  train  
4       D15  train  


In [9]:
IMG_SIZE = (256, 256)
BATCH_SIZE = 16

def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = preprocess_input(img)
    return img, label

def make_dataset(split_name, shuffle=False):
    subset = df[df['split'] == split_name]
    paths = subset['image_path'].values
    labels = subset['class_id'].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=42)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset('train', shuffle=True)
val_ds = make_dataset('val')
test_ds = make_dataset('test')

print("Datasets created successfully.")
for images, labels in train_ds.take(1):
    print("Batch image shape:", images.shape)
    print("Batch label shape:", labels.shape)

Datasets created successfully.
Batch image shape: (16, 256, 256, 3)
Batch label shape: (16,)


In [10]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
])
print("Augmentation layer ready.")

Augmentation layer ready.


In [11]:
base_model = MobileNetV2(
    input_shape=(256, 256, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

print("Base model loaded successfully.")
print("Number of layers in base model:", len(base_model.layers))
print("Output shape of base model:", base_model.output_shape)

/var/folders/qk/vk69rjr14bn26ykr7clkf9440000gn/T/ipykernel_97896/2042179500.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Base model loaded successfully.
Number of layers in base model: 154
Output shape of base model: (None, 8, 8, 1280)


In [12]:
num_classes = df['class_name'].nunique()

inputs = tf.keras.Input(shape=(256, 256, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_2 (Sequential)       │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [13]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Model compiled successfully.")

Model compiled successfully.


In [14]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint_path = "../models/best_model.keras"

callbacks = [
    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1
    )
]

print("Callbacks configured successfully.")

Callbacks configured successfully.


In [15]:
EPOCHS = 30

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/30


/Users/shalinirichhariya/Desktop/Wheat-Image-Classification/venv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step - accuracy: 0.4031 - loss: 1.2617
Epoch 1: val_accuracy improved from None to 0.51786, saving model to ../models/best_model.keras

Epoch 1: finished saving model to ../models/best_model.keras
49/49 ━━━━━━━━━━━━━━━━━━━━ 24s 373ms/step - accuracy: 0.4031 - loss: 1.2617 - val_accuracy: 0.5179 - val_loss: 0.9771
Epoch 2/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 285ms/step - accuracy: 0.4566 - loss: 1.0496
Epoch 2: val_accuracy did not improve from 0.51786
49/49 ━━━━━━━━━━━━━━━━━━━━ 18s 340ms/step - accuracy: 0.4566 - loss: 1.0496 - val_accuracy: 0.3512 - val_loss: 1.1142
Epoch 3/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 231ms/step - accuracy: 0.4503 - loss: 1.0257
Epoch 3: val_accuracy improved from 0.51786 to 0.55952, saving model to ../models/best_model.keras

Epoch 3: finished saving model to ../models/best_model.keras
49/49 ━━━━━━━━━━━━━━━━━━━━ 14s 282ms/step - accuracy: 0.4503 - loss: 1.0257 - val_accuracy: 0.5595 - val_loss: 0.9311
Epoch 4/30
49/49 ━━━━━━━━━━━━